# in-kind Process Template Visualizer
### Interactive graph explorer using `ipycytoscape`

Visualises the sorting workflow templates defined in `process_template.yaml`:  
step sequences, conditional branches, path types, category affinity, and preconditions.

**Setup:** run the install cell once, then restart the kernel.

In [4]:
import ipycytoscape
import ipywidgets as widgets
from IPython.display import display, HTML
import yaml, pathlib, re
from collections import defaultdict

In [5]:
# ── Template data ────────────────────────────────────────────────────────────
# Mirrors process_template.yaml instances. Edit here or load from YAML file.

STEP_META = {
    "take_photo":                    dict(label="Take photo",          family="data_capture",   cost=0.75, modality="camera"),
    "take_audio_note":               dict(label="Record audio note",   family="data_capture",   cost=0.5,  modality="audio"),
    "barcode_scan":                  dict(label="Scan barcode",        family="data_capture",   cost=0.25, modality="scan"),
    "assign_category":               dict(label="Assign category",     family="data_capture",   cost=0.5,  modality="form"),
    "ml_recognition":                dict(label="ML recognition",      family="data_capture",   cost=0.1,  modality="auto"),
    "review_ml_result":              dict(label="Review ML/barcode",   family="process_control",cost=0.4,  modality="confirm"),
    "assess_condition_clothing":     dict(label="Assess clothing",     family="data_capture",   cost=1.5,  modality="form"),
    "assess_condition_electronics":  dict(label="Assess electronics",  family="data_capture",   cost=2.5,  modality="form"),
    "perform_data_wipe":             dict(label="Data wipe (GDPR)",    family="data_capture",   cost=3.0,  modality="form"),
    "assess_condition_food":         dict(label="Assess food safety",  family="data_capture",   cost=0.75, modality="form"),
    "assess_condition_furniture":    dict(label="Assess furniture",    family="data_capture",   cost=2.0,  modality="form"),
    "assess_condition_books":        dict(label="Assess books",        family="data_capture",   cost=0.5,  modality="form"),
    "assess_condition_toys":         dict(label="Assess toys",         family="data_capture",   cost=1.0,  modality="form"),
    "assess_condition_personal_care":dict(label="Assess personal care",family="data_capture",   cost=0.5,  modality="form"),
    "assess_condition_mobility_aids":dict(label="Assess mobility aid", family="data_capture",   cost=3.0,  modality="form"),
    "confirm_disposal":              dict(label="Confirm disposal",    family="process_control",cost=0.3,  modality="confirm"),
    "flag_for_specialist_review":    dict(label="Flag for specialist", family="process_control",cost=0.5,  modality="form"),
    "assign_storage":                dict(label="Assign storage",      family="terminal",       cost=0.5,  modality="form",   postcondition="stored"),
    "set_category":                  dict(label="Set signal category", family="data_capture",   cost=0.3,  modality="form"),
    "set_signal_type":               dict(label="Set signal type",     family="data_capture",   cost=0.2,  modality="form"),
    "assess_need":                   dict(label="Assess need urgency", family="data_capture",   cost=1.0,  modality="form"),
    "set_attributes":                dict(label="Set attributes",      family="terminal",       cost=0.5,  modality="form",   postcondition="active"),
    "set_priority":                  dict(label="Set priority",        family="terminal",       cost=0.5,  modality="form",   postcondition="active"),
}

TEMPLATES = {
  "sort_manual_clothing": dict(
    label="Manual — clothing", path_type="manual_assessment", completeness="standard", duration=3.25,
    preconditions=[],
    affinity=dict(clothing="preferred", accessories="preferred", bedding_textiles="neutral"),
    steps=[
      dict(k="take_photo",               seq=1, opt=True,  cond=None),
      dict(k="assign_category",          seq=2, opt=False, cond=None),
      dict(k="assess_condition_clothing",seq=3, opt=False, cond=None),
      dict(k="take_audio_note",          seq=4, opt=True,  cond=None),
      dict(k="confirm_disposal",         seq=5, opt=True,  cond="condition=rejected"),
      dict(k="assign_storage",           seq=6, opt=False, cond=None),
    ]),
  "sort_manual_electronics": dict(
    label="Manual — electronics", path_type="manual_assessment", completeness="standard", duration=4.5,
    preconditions=[],
    affinity=dict(electronics="neutral"),
    steps=[
      dict(k="take_photo",                   seq=1, opt=True,  cond=None),
      dict(k="assign_category",              seq=2, opt=False, cond=None),
      dict(k="assess_condition_electronics", seq=3, opt=False, cond=None),
      dict(k="perform_data_wipe",            seq=4, opt=True,  cond="data_wipe=true"),
      dict(k="flag_for_specialist_review",   seq=5, opt=True,  cond="assessment=for_parts"),
      dict(k="confirm_disposal",             seq=6, opt=True,  cond="assessment=for_parts"),
      dict(k="assign_storage",               seq=7, opt=False, cond=None),
    ]),
  "sort_manual_food": dict(
    label="Manual — food", path_type="manual_assessment", completeness="standard", duration=1.5,
    preconditions=[],
    affinity=dict(food="neutral"),
    steps=[
      dict(k="assign_category",      seq=1, opt=False, cond=None),
      dict(k="assess_condition_food", seq=2, opt=False, cond=None),
      dict(k="confirm_disposal",      seq=3, opt=True,  cond="packaging=false"),
      dict(k="assign_storage",        seq=4, opt=False, cond=None),
    ]),
  "sort_manual_furniture": dict(
    label="Manual — furniture", path_type="manual_assessment", completeness="standard", duration=3.5,
    preconditions=[],
    affinity=dict(furniture="preferred", household="neutral"),
    steps=[
      dict(k="take_photo",                seq=1, opt=True,  cond=None),
      dict(k="assign_category",           seq=2, opt=False, cond=None),
      dict(k="assess_condition_furniture",seq=3, opt=False, cond=None),
      dict(k="take_audio_note",           seq=4, opt=True,  cond=None),
      dict(k="confirm_disposal",          seq=5, opt=True,  cond="condition=rejected"),
      dict(k="assign_storage",            seq=6, opt=False, cond=None),
    ]),
  "sort_manual_books": dict(
    label="Manual — books", path_type="manual_assessment", completeness="standard", duration=1.25,
    preconditions=[],
    affinity=dict(books="neutral"),
    steps=[
      dict(k="assign_category",      seq=1, opt=False, cond=None),
      dict(k="assess_condition_books",seq=2, opt=False, cond=None),
      dict(k="confirm_disposal",      seq=3, opt=True,  cond="condition=rejected"),
      dict(k="assign_storage",        seq=4, opt=False, cond=None),
    ]),
  "sort_manual_toys": dict(
    label="Manual — toys", path_type="manual_assessment", completeness="standard", duration=1.75,
    preconditions=[],
    affinity=dict(toys="preferred"),
    steps=[
      dict(k="assign_category",     seq=1, opt=False, cond=None),
      dict(k="assess_condition_toys",seq=2, opt=False, cond=None),
      dict(k="confirm_disposal",    seq=3, opt=True,  cond="condition=rejected"),
      dict(k="assign_storage",      seq=4, opt=False, cond=None),
    ]),
  "sort_manual_mobility_aids": dict(
    label="Manual — mobility aids", path_type="manual_assessment", completeness="standard", duration=5.0,
    preconditions=[],
    affinity=dict(mobility_aids="preferred"),
    steps=[
      dict(k="take_photo",                    seq=1, opt=True,  cond=None),
      dict(k="assign_category",               seq=2, opt=False, cond=None),
      dict(k="assess_condition_mobility_aids",seq=3, opt=False, cond=None),
      dict(k="flag_for_specialist_review",    seq=4, opt=True,  cond="condition=poor"),
      dict(k="confirm_disposal",              seq=5, opt=True,  cond="condition=rejected"),
      dict(k="assign_storage",                seq=6, opt=False, cond=None),
    ]),
  "sort_ml_electronics": dict(
    label="ML-assisted — electronics", path_type="ml_assisted", completeness="detailed", duration=3.5,
    preconditions=["Trained ML inference endpoint configured", "Model covers electronics category", "Photo capture enabled"],
    affinity=dict(electronics="preferred", clothing="neutral", furniture="neutral", food="discouraged", books="discouraged"),
    steps=[
      dict(k="take_photo",                   seq=1, opt=False, cond=None),
      dict(k="ml_recognition",               seq=2, opt=False, cond=None),
      dict(k="review_ml_result",             seq=3, opt=False, cond=None),
      dict(k="assess_condition_electronics", seq=4, opt=False, cond=None),
      dict(k="perform_data_wipe",            seq=5, opt=True,  cond="data_wipe=true"),
      dict(k="flag_for_specialist_review",   seq=6, opt=True,  cond="assessment=for_parts"),
      dict(k="confirm_disposal",             seq=7, opt=True,  cond="assessment=for_parts"),
      dict(k="assign_storage",               seq=8, opt=False, cond=None),
    ]),
  "sort_ml_clothing": dict(
    label="ML-assisted — clothing", path_type="ml_assisted", completeness="detailed", duration=3.65,
    preconditions=["Trained ML inference endpoint configured (clothing, confidence ≥ 0.85)", "Photo capture enabled"],
    affinity=dict(clothing="neutral", accessories="neutral"),
    steps=[
      dict(k="take_photo",               seq=1, opt=False, cond=None),
      dict(k="ml_recognition",           seq=2, opt=False, cond=None),
      dict(k="review_ml_result",         seq=3, opt=False, cond=None),
      dict(k="assess_condition_clothing",seq=4, opt=False, cond=None),
      dict(k="confirm_disposal",         seq=5, opt=True,  cond="condition=rejected"),
      dict(k="assign_storage",           seq=6, opt=False, cond=None),
    ]),
  "sort_barcode_food": dict(
    label="Barcode-first — food", path_type="barcode_first", completeness="detailed", duration=1.25,
    preconditions=["Barcode scanner configured", "Open Food Facts API enabled", "GS1 GEPIR lookup enabled"],
    affinity=dict(food="preferred", personal_care="preferred", household="neutral", clothing="discouraged", furniture="discouraged", mobility_aids="discouraged"),
    steps=[
      dict(k="barcode_scan",        seq=1, opt=False, cond=None),
      dict(k="review_ml_result",    seq=2, opt=False, cond=None),
      dict(k="assess_condition_food",seq=3, opt=False, cond=None),
      dict(k="confirm_disposal",    seq=4, opt=True,  cond="packaging=false"),
      dict(k="assign_storage",      seq=5, opt=False, cond=None),
    ]),
  "sort_barcode_books": dict(
    label="Barcode-first — books", path_type="barcode_first", completeness="detailed", duration=0.85,
    preconditions=["Barcode scanner configured", "Open Library / Google Books API enabled"],
    affinity=dict(books="preferred", electronics="neutral", clothing="discouraged"),
    steps=[
      dict(k="barcode_scan",          seq=1, opt=False, cond=None),
      dict(k="review_ml_result",      seq=2, opt=False, cond=None),
      dict(k="assess_condition_books",seq=3, opt=False, cond=None),
      dict(k="confirm_disposal",      seq=4, opt=True,  cond="condition=rejected"),
      dict(k="assign_storage",        seq=5, opt=False, cond=None),
    ]),
  "sort_rapid_triage": dict(
    label="Rapid triage", path_type="rapid_triage", completeness="minimal", duration=1.25,
    preconditions=[],
    affinity=dict(clothing="neutral", food="discouraged", mobility_aids="discouraged"),
    steps=[
      dict(k="assign_category", seq=1, opt=False, cond=None),
      dict(k="take_audio_note", seq=2, opt=True,  cond=None),
      dict(k="assign_storage",  seq=3, opt=False, cond=None),
    ]),
  "register_demand_signal": dict(
    label="Demand signal — standard", path_type="demand_signal_standard", completeness="standard", duration=2.0,
    preconditions=[],
    affinity={},
    steps=[
      dict(k="set_category",    seq=1, opt=False, cond=None),
      dict(k="set_signal_type", seq=2, opt=False, cond=None),
      dict(k="assess_need",     seq=3, opt=True,  cond=None),
      dict(k="set_attributes",  seq=4, opt=False, cond=None),
    ]),
  "register_demand_signal_urgent": dict(
    label="Demand signal — urgent", path_type="demand_signal_urgent", completeness="minimal", duration=1.0,
    preconditions=[],
    affinity={},
    steps=[
      dict(k="set_category",    seq=1, opt=False, cond=None),
      dict(k="set_signal_type", seq=2, opt=False, cond=None),
      dict(k="set_priority",    seq=3, opt=False, cond=None),
      dict(k="set_attributes",  seq=4, opt=False, cond=None),
    ]),
}

PATH_COLORS = {
    "manual_assessment":     "#1D9E75",
    "ml_assisted":           "#7F77DD",
    "barcode_first":         "#BA7517",
    "rapid_triage":          "#888780",
    "demand_signal_standard":"#378ADD",
    "demand_signal_urgent":  "#D85A30",
}
FAMILY_COLORS = {
    "data_capture":   "#1D9E75",
    "process_control":"#378ADD",
    "terminal":       "#D85A30",
}
FAMILY_BG = {
    "data_capture":   "#E1F5EE",
    "process_control":"#E6F1FB",
    "terminal":       "#FAECE7",
}
print(f"Loaded {len(TEMPLATES)} templates, {len(STEP_META)} step types")

Loaded 14 templates, 23 step types


In [6]:
# ── Graph builders ────────────────────────────────────────────────────────────

def slug(tpl_id, i, step_key):
    return f"{tpl_id}__{i}__{step_key}"

def build_template_graph(tpl_id):
    """Build ipycytoscape nodes + edges for a single template."""
    tpl = TEMPLATES[tpl_id]
    nodes, edges = [], []
    start_id = f"{tpl_id}__START"
    nodes.append({"data": {"id": start_id, "label": "", "type": "start"}})
    prev = start_id
    for i, s in enumerate(tpl["steps"]):
        m = STEP_META.get(s["k"], {})
        nid = slug(tpl_id, i, s["k"])
        nodes.append({"data": {
            "id": nid,
            "label": m.get("label", s["k"]),
            "family": m.get("family", "data_capture"),
            "optional": s["opt"],
            "cond": s.get("cond") or "",
            "cost": m.get("cost", 0),
            "modality": m.get("modality", ""),
            "postcondition": m.get("postcondition", ""),
            "seq": s["seq"],
            "step_key": s["k"],
        }})
        edges.append({"data": {
            "source": prev,
            "target": nid,
            "conditional": bool(s.get("cond")),
            "label": s.get("cond") or "",
        }})
        prev = nid
    return nodes, edges

def make_cytoscape(tpl_id):
    """Return a configured CytoscapeWidget for the given template."""
    nodes, edges = build_template_graph(tpl_id)
    cy = ipycytoscape.CytoscapeWidget()
    cy.graph.add_graph_from_dict({"nodes": nodes, "edges": edges})
    cy.set_style([
        {"selector": "node", "css": {
            "label": "data(label)",
            "text-valign": "center", "text-halign": "center",
            "text-wrap": "wrap", "text-max-width": "110px",
            "font-size": "11px",
            "width": "120px", "height": "40px",
            "shape": "roundrectangle",
            "background-color": "mapData(family,data_capture,terminal,#E1F5EE,#FAECE7)",
            "border-width": 1.5,
            "border-color": "#aaa",
            "color": "#222",
        }},
        {"selector": "node[family='data_capture']",    "css": {"background-color": "#E1F5EE", "border-color": "#1D9E75"}},
        {"selector": "node[family='process_control']", "css": {"background-color": "#E6F1FB", "border-color": "#378ADD"}},
        {"selector": "node[family='terminal']",        "css": {"background-color": "#FAECE7", "border-color": "#D85A30"}},
        {"selector": "node[?optional]",                "css": {"border-style": "dashed", "opacity": 0.75}},
        {"selector": "node[type='start']",             "css": {"shape": "ellipse", "width": "18px", "height": "18px",
                                                                "background-color": "#888", "border-width": 0, "label": ""}},
        {"selector": "edge",                           "css": {
            "width": 1.5,
            "line-color": "#bbb",
            "target-arrow-color": "#bbb",
            "target-arrow-shape": "triangle",
            "curve-style": "bezier",
        }},
        {"selector": "edge[?conditional]",             "css": {
            "line-color": "#BA7517",
            "target-arrow-color": "#BA7517",
            "line-style": "dashed",
            "label": "data(label)",
            "font-size": "9px",
            "color": "#BA7517",
        }},
        {"selector": ":selected",                      "css": {"border-color": "#7F77DD", "border-width": 2.5}},
    ])
    cy.set_layout(name="dagre", rankDir="LR", nodeSep=28, rankSep=55, padding=20)
    cy.layout.run()
    return cy

print("Graph builders ready")

Graph builders ready


# ── Affinity matrix ─────────────────────────────────────────────────────────

In [7]:
# ── Affinity matrix as a styled HTML table ───────────────────────────────────

CATEGORIES = ["clothing","electronics","food","furniture","books","toys",
               "mobility_aids","accessories","personal_care","household"]
PATH_TYPES  = ["manual_assessment","ml_assisted","barcode_first","rapid_triage"]
PT_LABEL    = {"manual_assessment":"Manual","ml_assisted":"ML-assisted",
               "barcode_first":"Barcode-first","rapid_triage":"Rapid triage"}

PILL = {
    "preferred":   ("★","#085041","#E1F5EE"),
    "neutral":     ("○","#5F5E5A","#F1EFE8"),
    "discouraged": ("✗","#712B13","#FAECE7"),
}

def affinity_table():
    rows = ["<table style='border-collapse:collapse;font-size:12px'>"]
    rows.append("<tr><th style='padding:6px 10px;text-align:left;border:1px solid #ddd'>Category</th>"
                + "".join(f"<th style='padding:6px 10px;border:1px solid #ddd;color:{PATH_COLORS[pt]}'>{PT_LABEL[pt]}</th>"
                          for pt in PATH_TYPES) + "</tr>")
    for cat in CATEGORIES:
        row = [f"<td style='padding:5px 10px;border:1px solid #ddd;font-weight:500'>{cat}</td>"]
        for pt in PATH_TYPES:
            best = None
            for tid, t in TEMPLATES.items():
                if t["path_type"] == pt and cat in t["affinity"]:
                    a = t["affinity"][cat]
                    if best is None or ["preferred","neutral","discouraged"].index(a) < ["preferred","neutral","discouraged"].index(best[0]):
                        best = (a, t["duration"])
            if best:
                icon, tc, bg = PILL[best[0]]
                row.append(f"<td style='padding:5px 10px;border:1px solid #ddd'>"
                            f"<span style='background:{bg};color:{tc};padding:2px 8px;border-radius:10px;font-weight:500'>{icon} {best[0]}</span>"
                            f"<br><span style='font-size:10px;color:#888'>{best[1]:.1f} min</span></td>")
            else:
                row.append("<td style='padding:5px 10px;border:1px solid #ddd;color:#ccc;font-size:10px'>—</td>")
        rows.append("<tr>" + "".join(row) + "</tr>")
    rows.append("</table>")
    return "".join(rows)

display(HTML("<h4>Category × path type affinity matrix</h4>" + affinity_table()))
display(HTML("<p style='font-size:11px;color:#888'>★ preferred &nbsp;○ neutral &nbsp;✗ discouraged — best template per cell shown</p>"))

Category,Manual,ML-assisted,Barcode-first,Rapid triage
clothing,★ preferred3.2 min,○ neutral3.5 min,✗ discouraged1.2 min,○ neutral1.2 min
electronics,○ neutral4.5 min,★ preferred3.5 min,○ neutral0.8 min,—
food,○ neutral1.5 min,✗ discouraged3.5 min,★ preferred1.2 min,✗ discouraged1.2 min
furniture,★ preferred3.5 min,○ neutral3.5 min,✗ discouraged1.2 min,—
books,○ neutral1.2 min,✗ discouraged3.5 min,★ preferred0.8 min,—
toys,★ preferred1.8 min,—,—,—
mobility_aids,★ preferred5.0 min,—,✗ discouraged1.2 min,✗ discouraged1.2 min
accessories,★ preferred3.2 min,○ neutral3.6 min,—,—
personal_care,—,—,★ preferred1.2 min,—
household,○ neutral3.5 min,—,○ neutral1.2 min,—


# ── Preconditions table ────────────────────────────────────────────────────

In [8]:
# ── Preconditions summary ─────────────────────────────────────────────────────

rows = ["<table style='border-collapse:collapse;font-size:12px;width:100%'>",
        "<tr><th style='padding:6px 10px;text-align:left;border:1px solid #ddd'>Template</th>"
        "<th style='padding:6px 10px;text-align:left;border:1px solid #ddd'>Path type</th>"
        "<th style='padding:6px 10px;text-align:left;border:1px solid #ddd'>Preconditions</th>"
        "<th style='padding:6px 10px;text-align:right;border:1px solid #ddd'>Est. min</th></tr>"]

for tid, t in TEMPLATES.items():
    precond_html = ("<ul style='margin:0;padding-left:14px'>" +
                    "".join(f"<li>{p}</li>" for p in t["preconditions"]) +
                    "</ul>") if t["preconditions"] else "<span style='color:#aaa'>none</span>"
    col = PATH_COLORS.get(t["path_type"], "#888")
    rows.append(
        f"<tr>"
        f"<td style='padding:5px 10px;border:1px solid #ddd;font-weight:500'>{t['label']}</td>"
        f"<td style='padding:5px 10px;border:1px solid #ddd'>"
        f"<span style='color:{col};font-weight:500'>{t['path_type']}</span></td>"
        f"<td style='padding:5px 10px;border:1px solid #ddd;font-size:11px'>{precond_html}</td>"
        f"<td style='padding:5px 10px;border:1px solid #ddd;text-align:right'>{t['duration']:.2f}</td>"
        f"</tr>"
    )
rows.append("</table>")
display(HTML("<h4>Template preconditions</h4>" + "".join(rows)))

Template,Path type,Preconditions,Est. min
Manual — clothing,manual_assessment,none,3.25
Manual — electronics,manual_assessment,none,4.50
Manual — food,manual_assessment,none,1.50
Manual — furniture,manual_assessment,none,3.50
Manual — books,manual_assessment,none,1.25
Manual — toys,manual_assessment,none,1.75
Manual — mobility aids,manual_assessment,none,5.00
ML-assisted — electronics,ml_assisted,Trained ML inference endpoint configuredModel covers electronics categoryPhoto capture enabled,3.50
ML-assisted — clothing,ml_assisted,"Trained ML inference endpoint configured (clothing, confidence ≥ 0.85)Photo capture enabled",3.65
Barcode-first — food,barcode_first,Barcode scanner configuredOpen Food Facts API enabledGS1 GEPIR lookup enabled,1.25


## Interactive single-template explorer
Select a template from the dropdown to see its step graph, preconditions, and step table.

In [9]:
# ── Single template explorer ─────────────────────────────────────────────────

from ipywidgets import interact, Dropdown, Output

template_options = [(t["label"], tid) for tid, t in TEMPLATES.items()]

info_out = Output()

def show_template(template_id):
    tpl = TEMPLATES[template_id]
    cy = make_cytoscape(template_id)
    cy.layout.run()

    # precondition banner
    prec_html = ""
    if tpl["preconditions"]:
        items = "".join(f"<li>{p}</li>" for p in tpl["preconditions"])
        prec_html = (f"<div style='background:#FAEEDA;border-left:3px solid #BA7517;"
                     f"padding:8px 12px;margin:6px 0;border-radius:4px;font-size:12px'>"
                     f"<strong>Preconditions</strong><ul style='margin:4px 0 0;padding-left:16px'>{items}</ul></div>")

    col = PATH_COLORS.get(tpl["path_type"], "#888")
    header = (f"<div style='margin:6px 0;font-size:13px'>"
              f"<span style='color:{col};font-weight:500'>{tpl['path_type']}</span>"
              f" &nbsp;|&nbsp; completeness: <strong>{tpl['completeness']}</strong>"
              f" &nbsp;|&nbsp; est. <strong>{tpl['duration']:.2f} min</strong>"
              f" &nbsp;|&nbsp; {len(tpl['steps'])} steps</div>")

    with info_out:
        info_out.clear_output()
        display(HTML(header + prec_html))
        display(cy)

        # step table
        step_rows = ["<table style='border-collapse:collapse;font-size:11px;margin-top:8px'>",
                     "<tr><th style='padding:4px 8px;border:1px solid #ddd'>#</th>"
                     "<th style='padding:4px 8px;border:1px solid #ddd'>Step</th>"
                     "<th style='padding:4px 8px;border:1px solid #ddd'>Family</th>"
                     "<th style='padding:4px 8px;border:1px solid #ddd'>Modality</th>"
                     "<th style='padding:4px 8px;border:1px solid #ddd;text-align:right'>Cost</th>"
                     "<th style='padding:4px 8px;border:1px solid #ddd'>Optional?</th>"
                     "<th style='padding:4px 8px;border:1px solid #ddd'>Condition</th></tr>"]
        for s in tpl["steps"]:
            m = STEP_META.get(s["k"], {})
            fc = FAMILY_COLORS.get(m.get("family",""), "#888")
            step_rows.append(
                f"<tr><td style='padding:4px 8px;border:1px solid #ddd'>{s['seq']}</td>"
                f"<td style='padding:4px 8px;border:1px solid #ddd'>{m.get('label', s['k'])}</td>"
                f"<td style='padding:4px 8px;border:1px solid #ddd;color:{fc}'>{m.get('family','')}</td>"
                f"<td style='padding:4px 8px;border:1px solid #ddd'>{m.get('modality','')}</td>"
                f"<td style='padding:4px 8px;border:1px solid #ddd;text-align:right'>{m.get('cost',0)}</td>"
                f"<td style='padding:4px 8px;border:1px solid #ddd'>{'yes' if s['opt'] else ''}</td>"
                f"<td style='padding:4px 8px;border:1px solid #ddd;color:#BA7517;font-size:10px'>{s.get('cond') or ''}</td>"
                f"</tr>")
        step_rows.append("</table>")
        display(HTML("".join(step_rows)))

w = Dropdown(options=template_options, description="Template:", layout={"width": "380px"})
display(widgets.VBox([w, info_out]))
w.observe(lambda change: show_template(change["new"]) if change["name"]=="value" else None)
show_template(w.value)

AttributeError: 'Graph' object has no attribute 'add_graph_from_dict'

## All templates overview
One node per template. **Dashed border** = has preconditions. Colour = path type.

In [ ]:
# ── Path-type overview: all templates as nodes grouped by type ───────────────

import ipycytoscape

cy_ov = ipycytoscape.CytoscapeWidget()

ov_nodes, ov_edges = [], []
# Group nodes in a grid by path_type
type_order = ["manual_assessment","ml_assisted","barcode_first","rapid_triage",
               "demand_signal_standard","demand_signal_urgent"]
type_counts = {pt: 0 for pt in type_order}

for tid, t in TEMPLATES.items():
    pt = t["path_type"]
    col = PATH_COLORS.get(pt, "#888")
    ov_nodes.append({"data": {
        "id": tid,
        "label": t["label"],
        "path_type": pt,
        "completeness": t["completeness"],
        "duration": t["duration"],
        "has_preconditions": len(t["preconditions"]) > 0,
    }})

# Variant edges (same path-type family)
variant_map = {}
for tid, t in TEMPLATES.items():
    if t.get("path_type") in ["manual_assessment"]:
        variant_map.setdefault(t["path_type"], []).append(tid)

cy_ov.graph.add_graph_from_dict({"nodes": ov_nodes, "edges": ov_edges})
cy_ov.set_style([
    {"selector": "node", "css": {
        "label": "data(label)",
        "text-valign": "center", "text-halign": "center",
        "text-wrap": "wrap", "text-max-width": "130px",
        "font-size": "11px", "width": "140px", "height": "48px",
        "shape": "roundrectangle", "border-width": 2,
    }},
    *[{"selector": f"node[path_type='{pt}']", "css": {
        "background-color": col + "22",
        "border-color": col,
        "color": col,
    }} for pt, col in PATH_COLORS.items()],
    {"selector": "node[?has_preconditions]", "css": {
        "border-style": "dashed",
    }},
])
cy_ov.set_layout(name="grid", cols=4, padding=20)
cy_ov.layout.run()

display(HTML(
    "<h4>All templates (dashed border = has preconditions)</h4>"
    + "".join(f"<span style='display:inline-block;margin:3px 6px;font-size:11px'>"
              f"<span style='display:inline-block;width:10px;height:10px;border-radius:2px;"
              f"background:{col};margin-right:3px'></span>{PT_LABEL.get(pt,pt)}</span>"
              for pt, col in PATH_COLORS.items())
))
display(cy_ov)

## Path cost comparison
`C_s(p)` = sum of default step costs. Orgs can override per-step scalars.

In [ ]:
# ── Path cost comparison chart (matplotlib) ──────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(11, 5))
ids = list(TEMPLATES.keys())
labels = [TEMPLATES[t]["label"] for t in ids]
durations = [TEMPLATES[t]["duration"] for t in ids]
colors = [PATH_COLORS.get(TEMPLATES[t]["path_type"], "#888") for t in ids]
completeness_marker = {"minimal": "○", "standard": "●", "detailed": "◆"}

bars = ax.barh(range(len(ids)), durations, color=colors, alpha=0.82, height=0.6)
ax.set_yticks(range(len(ids)))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Estimated duration (minutes)", fontsize=11)
ax.set_title("Template path cost comparison  C_s(p)", fontsize=12, pad=10)
ax.invert_yaxis()
ax.axvline(x=2, color="#aaa", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(2.05, -0.7, "2 min", fontsize=8, color="#aaa")

for i, (tid, dur) in enumerate(zip(ids, durations)):
    c = completeness_marker[TEMPLATES[tid]["completeness"]]
    ax.text(dur + 0.05, i, f"{dur:.2f}  {c}", va="center", fontsize=9)

legend_elems = [mpatches.Patch(color=col, alpha=0.82, label=pt.replace("_"," "))
                for pt, col in PATH_COLORS.items()]
completeness_elems = [plt.Line2D([0],[0], marker=m, color="gray", linestyle="None",
                                  markersize=8, label=f"completeness: {k}")
                      for k, m in completeness_marker.items()]
ax.legend(handles=legend_elems + completeness_elems, fontsize=9,
          loc="lower right", framealpha=0.85)
plt.tight_layout()
plt.show()

## Notes on the data model

- **Templates are pre-selected per org** in the Django admin (`OrgProcessConfig`). There is no runtime capability check — the admin enabling a template is responsible for meeting its preconditions.
- **Preconditions** are human-readable annotations on the template, surfaced in the admin UI. They are informational, not enforced by the engine.
- **Category affinity** (`preferred` / `neutral` / `discouraged`) is an expected-value prior: the engine weights feasible paths by affinity when multiple paths are available for an item. It does not gate selection.
- **Completeness tier** determines which `v_λ` scoring functions are available at the match step downstream. A `minimal` path omits the match dimension entirely until a re-sort episode is launched.
- See `process_template.yaml` for the full YAML source and `domain_model.md` for the conceptual overview.